# Serialize a Python feature set for Rust

This notebook reads `trades.csv`, creates grouped indicators for every symbol in the file, writes `feature_set.json`, and verifies that Python can reload and compile the artifact.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import fiml

In [ ]:
working_dir = Path.cwd()
notebook_dir = working_dir if (working_dir / "trades.csv").exists() else working_dir / "notebooks"
trades_path = notebook_dir / "trades.csv"
feature_set_path = notebook_dir / "feature_set.json"

assert trades_path.exists(), f"Could not find trades.csv from {working_dir}"
trades = pd.read_csv(trades_path)
symbols = trades["symbol"].drop_duplicates().tolist()

assert symbols, "trades.csv must contain at least one symbol"
trades

In [ ]:
feature_set = fiml.FeatureSet()
for symbol in symbols:
    feature_set.sma(symbol, [2, 4], source="trade_price")
    feature_set.ema(symbol, [2], source="trade_price")
    feature_set.trade_count_timed(symbol, aggregation="1ms", window="60s")

feature_set.day_of_week()

assert feature_set.indicator_count() == len(symbols) * 3 + 1
assert feature_set.output_count() == len(symbols) * 4 + 1

In [ ]:
serialized = feature_set.to_json()
pretty_json = json.dumps(json.loads(serialized), indent=2)
feature_set_path.write_text(pretty_json + "\n", encoding="utf-8")

print(f"Wrote {feature_set_path}")
print(pretty_json)

In [ ]:
restored = fiml.FeatureSet.from_json(feature_set_path.read_text(encoding="utf-8"))
extractor = fiml.FeatureExtractor(restored)
features = extractor.compute_features(trades)

assert restored.indicator_count() == feature_set.indicator_count()
assert restored.output_count() == feature_set.output_count()
assert list(features.columns[2:]) == extractor.feature_names()
features

Load the generated artifact from the repository root with:

```bash
cargo run -p fiml --example feature_set_from_json --features serde
```